In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "ollama:gemma4:e4b",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='131737f7-ffed-4fb4-a3e8-f8d008604531'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-07T10:22:06.122102Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7262658875, 'load_duration': 200287750, 'prompt_eval_count': 83, 'prompt_eval_duration': 6376827291, 'eval_count': 21, 'eval_duration': 672072834, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6776-98c9-7de1-bd7d-ca8b4ea2ec97-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': '18afeb54-4e72-4689-bd3a-1a0b7bf47b42', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 21, 'total_tokens': 104}),
              ToolMessage(content='Successfully updated favourite colour', name='

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='79f9ebb9-0f06-4e6c-ab3a-850aa7d904a1'),
              AIMessage(content="I'm doing well, thank you for asking! How are you today?", additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-07T10:23:54.749249Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9023180708, 'load_duration': 189575666, 'prompt_eval_count': 84, 'prompt_eval_duration': 1822003166, 'eval_count': 208, 'eval_duration': 6942580496, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6778-3a3a-7741-8150-1ae7dcefcbf2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 208, 'total_tokens': 292})]}


## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "ollama:gemma4:e4b",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='3f59fb17-9605-4bf2-b147-0e052ba88633'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-07T10:24:27.518941Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9792791333, 'load_duration': 206040250, 'prompt_eval_count': 118, 'prompt_eval_duration': 1199124542, 'eval_count': 246, 'eval_duration': 8297728669, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6778-b73a-7270-aba7-565f2087f99a-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'aa5ae61f-00b2-4bba-8f82-ce5dc06d5889', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 118, 'output_tokens': 246, 'total_tokens': 364}),
              ToolMessage(content='Successfully updated favourite colour', n

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='3f59fb17-9605-4bf2-b147-0e052ba88633'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-07T10:24:27.518941Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9792791333, 'load_duration': 206040250, 'prompt_eval_count': 118, 'prompt_eval_duration': 1199124542, 'eval_count': 246, 'eval_duration': 8297728669, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6778-b73a-7270-aba7-565f2087f99a-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'aa5ae61f-00b2-4bba-8f82-ce5dc06d5889', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 118, 'output_tokens': 246, 'total_tokens': 364}),
              ToolMessage(content='Successfully updated favourite colour', n